In [3]:
import pandas as pd
import numpy as np
from pathlib import Path 

frequency = 64
path = Path(f"data/raw/dreamt/data_{frequency}Hz")

COLS_TO_DROP = [
    "TIMESTAMP",
    "IBI",
    "Obstructive_Apnea",
    "Central_Apnea",
    "Hypopnea",
    "Multiple_Events",
]
nb_users_max = 25
X_all_patients = []
y_all_patients = []
patient_file_list = [f for f in path.iterdir() if f.is_file()]
for patient_id in range(nb_users_max):
    patient_file = patient_file_list.pop() 
    print(patient_file)
    df = pd.read_csv(patient_file)
    df["Sleep_Stage"] = df["Sleep_Stage"].replace("P", "W")
    df = df.drop(
                columns=COLS_TO_DROP
            )
    df = df[df["Sleep_Stage"] != "Missing"]
    y_all_patients.append(df.Sleep_Stage.to_numpy())
    X_all_patients.append(df.drop(columns=["Sleep_Stage"]).to_numpy())
    df["patient_id"] = patient_id + 1 



data/raw/dreamt/data_64Hz/S009_whole_df.csv
data/raw/dreamt/data_64Hz/S014_whole_df.csv
data/raw/dreamt/data_64Hz/S018_whole_df.csv
data/raw/dreamt/data_64Hz/S023_whole_df.csv
data/raw/dreamt/data_64Hz/S003_whole_df.csv
data/raw/dreamt/data_64Hz/S026_whole_df.csv
data/raw/dreamt/data_64Hz/S015_whole_df.csv
data/raw/dreamt/data_64Hz/S007_whole_df.csv
data/raw/dreamt/data_64Hz/S027_whole_df.csv
data/raw/dreamt/data_64Hz/S028_whole_df.csv
data/raw/dreamt/data_64Hz/S011_whole_df.csv
data/raw/dreamt/data_64Hz/S020_whole_df.csv
data/raw/dreamt/data_64Hz/S002_whole_df.csv
data/raw/dreamt/data_64Hz/S010_whole_df.csv
data/raw/dreamt/data_64Hz/S012_whole_df.csv
data/raw/dreamt/data_64Hz/S029_whole_df.csv
data/raw/dreamt/data_64Hz/S024_whole_df.csv
data/raw/dreamt/data_64Hz/S021_whole_df.csv
data/raw/dreamt/data_64Hz/S008_whole_df.csv
data/raw/dreamt/data_64Hz/S017_whole_df.csv
data/raw/dreamt/data_64Hz/S013_whole_df.csv
data/raw/dreamt/data_64Hz/S005_whole_df.csv
data/raw/dreamt/data_64Hz/S016_w

In [4]:
WINDOWS_SEC = 30
FS = 64

window_samples = FS * WINDOWS_SEC

X_bvp_patients = []
X_acc_patients = []
X_eda_temp_patients = []
X_hr_patients = []
y_patients = []

for patient in range(len(X_all_patients)):
    X_bvp = []
    X_acc = []
    X_eda_temp = []
    X_hr = []
    y = []
    data = X_all_patients[patient]
    T = data.shape[0]
    n_windows = T // window_samples
    for i in range(n_windows):
        start = i * window_samples
        end = start + window_samples
        # 1920, 
        X_bvp.append(data[start:end,0])
        # 960
        X_acc.append(data[start:end:2, 1:4])
        # 120
        X_eda_temp.append(data[start:end:16, 4:6])
        # 30
        X_hr.append(data[start:end:64, 6])
        #1
        y.append(y_all_patients[patient][start])
    
    X_bvp_patients.append(np.stack(X_bvp))
    X_acc_patients.append(np.stack(X_acc))
    X_hr_patients.append(np.stack(X_hr))
    X_eda_temp_patients.append(np.stack(X_eda_temp))
    y_patients.append(np.array(y))
        


In [5]:
X_bvp_train = []
X_bvp_test = []

X_acc_train = []
X_acc_test = []

X_eda_temp_train = []
X_eda_temp_test = []

X_hr_train = []
X_hr_test = []

y_train = []
y_test = []

test_size = 0.2

for patient in range(len(X_bvp_patients)):
    dataset_len = len(X_bvp_patients[patient])   
    split = 1 - int(dataset_len * test_size)
    X_bvp_train.append(X_bvp_patients[patient][:split])
    X_bvp_test.append(X_bvp_patients[patient][split:])
                      

    X_acc_train.append(X_acc_patients[patient][:split])
    X_acc_test.append(X_acc_patients[patient][split:])

                     
    X_eda_temp_train.append(X_eda_temp_patients[patient][:split])
    X_eda_temp_test.append(X_eda_temp_patients[patient][split:])
                       
    
    X_hr_train.append(X_hr_patients[patient][:split])
    X_hr_test.append(X_hr_patients[patient][split:])
                      
    y_train.append(y_patients[patient][:split])
    y_test.append(y_patients[patient][split:])



X_bvp_train =np.concatenate(X_bvp_train)
X_bvp_test =np.concatenate(X_bvp_test)

X_acc_train =np.concatenate(X_acc_train)
X_acc_test =np.concatenate(X_acc_test)

X_eda_temp_train =np.concatenate(X_eda_temp_train)
X_eda_temp_test =np.concatenate(X_eda_temp_test)

X_hr_train =np.concatenate(X_hr_train)
X_hr_test =np.concatenate(X_hr_test)

y_train = np.concatenate(y_train)
y_test = np.concatenate(y_test)


print(X_bvp_train.shape)
X_bvp_train = np.expand_dims(X_bvp_train, axis=1)
print(f"here {X_eda_temp_train.shape}")
X_eda_temp_train = np.permute_dims(X_eda_temp_train, axes=[0,2,1])
print(X_acc_train.shape)
X_acc_train = np.permute_dims(X_acc_train, axes=[0,2,1])

(21627, 1920)
here (21627, 120, 2)
(21627, 960, 3)


In [6]:
X_bvp_test      = np.expand_dims(X_bvp_test, axis=1)
X_eda_temp_test = np.permute_dims(X_eda_temp_test, axes=[0,2,1])
X_acc_test      = np.permute_dims(X_acc_test, axes=[0,2,1])

In [7]:
from sklearn.preprocessing import LabelEncoder

lb = LabelEncoder()
y_train_encoded = lb.fit_transform(y_train)
y_test_encoded = lb.transform(y_test)

y_train_encoded

array([4, 4, 4, ..., 1, 1, 1], shape=(21627,))

In [ ]:

import torch 
import torch.nn as nn 


class Residual(nn.Module):
    def __init__(self, input_size, hidden_size, kernel_size = 3, dilation_n = 3):
        super().__init__()
        self.blocks = nn.ModuleList()
        original_input_size = input_size
        for i in range(dilation_n): 
            dilation = 2 ** i 
            self.blocks.append(nn.Sequential(
            nn.ZeroPad1d(padding=((kernel_size -1) * dilation,0)),
            nn.Conv1d(in_channels=input_size, out_channels =hidden_size, kernel_size=kernel_size, 
                                dilation=dilation),
            nn.BatchNorm1d(hidden_size),
            nn.ReLU()
            ))
            input_size = hidden_size

        self.drop_out = nn.Dropout1d(0.5)
        self.align = nn.Conv1d(original_input_size, hidden_size, 1)  \
                if original_input_size != hidden_size else nn.Identity()
 


    def forward(self, input):
        x = input 
        for block in self.blocks:
            x = block(x)

        x = self.drop_out(x)

        return self.align(input) + x 
    


class MultiTCN(nn.Module):
    def __init__(self):
        super().__init__()
        self.bc_bvp = nn.BatchNorm1d(1)
        self.bc_acc = nn.BatchNorm1d(3)
        self.bc_temp = nn.BatchNorm1d(2)
        self.bc_hr = nn.BatchNorm1d(1)

        self.bvp_block = Residual(1, 32, dilation_n=6)
        self.acc_block = Residual(3, 32, dilation_n=5)
        self.eda_temp_block = Residual(2, 32, dilation_n=5)
        self.hr_block = Residual(1, 16, dilation_n=4)

        target_length = 30


        self.avg_bvp = nn.AvgPool1d(kernel_size=1920 // target_length)
        self.avg_acc = nn.AvgPool1d(kernel_size=960 // target_length)    
        self.avg_temp = nn.AvgPool1d(kernel_size=120 // target_length)   

        self.fc = nn.Sequential(
            nn.Linear(32 * target_length * 3 + 30, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 5),
        ) 

        self.init_weights()

    def init_weights(self):
        for layer in self.modules():
            if isinstance(layer, (nn.Conv1d, nn.Linear)):
                nn.init.kaiming_normal_(layer.weight)
                nn.init.zeros_(layer.bias)
            
    def forward(self, x_bvp, x_acc, x_eda_temp, x_hr):
        x_hr = x_hr.unsqueeze(1)
        x_bvp  = self.bc_bvp(x_bvp)
        x_acc = self.bc_acc(x_acc)
        x_eda_temp = self.bc_temp(x_eda_temp)
        x_hr = self.bc_hr(x_hr)
        out_bvp = self.bvp_block(x_bvp) #32, 1920
        out_acc = self.acc_block(x_acc) #32, 960
        out_eda_temp = self.eda_temp_block(x_eda_temp) #32, 120
        out_hr = self.hr_block(x_hr)

        out_bvp = self.avg_bvp(out_bvp) #32, 30
        out_acc = self.avg_acc(out_acc) #32, 30 
        out_eda_temp = self.avg_temp(out_eda_temp) #32, 30 
        merged = torch.cat([out_bvp, out_acc, out_eda_temp, x_hr], dim=1).flatten(1)

        return self.fc(merged)

In [10]:
from torch.utils.data import Dataset, DataLoader
class DreamtDataset(Dataset):
    def __init__(self, X_bvp, X_acc, X_eda_temp, X_hr, y):
        super().__init__()
        self.X_bvp      = X_bvp
        self.X_acc      = X_acc
        self.X_eda_temp = X_eda_temp
        self.X_hr       = X_hr
        self.y          = y

    def __getitem__(self, index):
        return (
            self.X_bvp[index],
            self.X_acc[index],
            self.X_eda_temp[index],
            self.X_hr[index],
            self.y[index],
        )

    def __len__(self):
        return len(self.X_bvp)



X_bvp_train      = torch.FloatTensor(X_bvp_train)
X_acc_train      = torch.FloatTensor(X_acc_train)
X_eda_temp_train = torch.FloatTensor(X_eda_temp_train)
X_hr_train       = torch.FloatTensor(X_hr_train)
y_train_encoded  = torch.LongTensor(y_train_encoded)  

train_ds = DreamtDataset(X_bvp_train, X_acc_train, X_eda_temp_train, X_hr_train, y_train_encoded)
train_dl = DataLoader(train_ds, batch_size=128, shuffle=True)

In [11]:
from sklearn.utils.class_weight import compute_class_weight

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

classes = np.unique(y_train_encoded.numpy())
weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_train_encoded.numpy())
weights = torch.FloatTensor(weights).to(DEVICE)

criterion = nn.CrossEntropyLoss(weight=weights, reduction="sum")

In [ ]:
from tqdm import tqdm



X_bvp_test       = torch.FloatTensor(X_bvp_test)
X_acc_test       = torch.FloatTensor(X_acc_test)
X_eda_temp_test  = torch.FloatTensor(X_eda_temp_test)
X_hr_test        = torch.FloatTensor(X_hr_test)
y_test_encoded   = torch.LongTensor(y_test_encoded)

test_ds = DreamtDataset(X_bvp_test, X_acc_test, X_eda_temp_test, X_hr_test, y_test_encoded)
test_dl = DataLoader(test_ds, batch_size=1024)


def train_model(model, train_dl, epochs, weights= None, lr=0.001, device=torch.device("cpu")):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    if weights is None:
        criterion = nn.CrossEntropyLoss(reduction="sum")
    else: 
        criterion = nn.CrossEntropyLoss(weight=weights, reduction="sum")
    for epoch in tqdm(range(epochs)):
        model.train()
        empirical_risk = 0.0
        for x_bvp, x_acc, x_eda_temp, x_hr, y in train_dl:
            x_bvp = x_bvp.to(device)
            x_acc = x_acc.to(device)
            x_eda_temp = x_eda_temp.to(device)
            x_hr = x_hr.to(device)
            y = y.to(device)
            optimizer.zero_grad()
            pred = model(x_bvp, x_acc, x_eda_temp, x_hr)
            loss = criterion(pred, y)
            loss.backward()
            optimizer.step()
            empirical_risk += loss.item()

        empirical_risk /= len(train_dl.dataset)
        print("Train loss: %.3f" % (empirical_risk))
        if epoch % 2 == 0:
            y_true, y_pred = test_model(model, test_dl, DEVICE)

def test_model(model, test_dl, device=torch.device("cpu")):
    criterion = nn.CrossEntropyLoss(reduction="sum")
    model.eval()
    generalization_error = 0.0
    correct = 0
    all_preds = []
    all_targets = []

    with torch.no_grad():
        for x_bvp, x_acc, x_eda_temp, x_hr, y in test_dl:
            x_bvp = x_bvp.to(device)
            x_acc = x_acc.to(device)
            x_eda_temp = x_eda_temp.to(device)
            x_hr = x_hr.to(device)
            y = y.to(device)
            logits = model(x_bvp, x_acc, x_eda_temp, x_hr)
            loss = criterion(logits, y)
            pred = torch.argmax(logits, dim=1)
            correct += (pred == y).sum().item()
            generalization_error += loss.item()
            all_preds.append(pred.cpu())
            all_targets.append(y.cpu())

        y_pred = torch.cat(all_preds).numpy()
        y_true = torch.cat(all_targets).numpy()
        generalization_error /= len(test_dl.dataset)
        accuracy = correct / len(test_dl.dataset)
        print(
            "Generalization Error: %.3f, Accuracy %.3f"
            % (generalization_error, accuracy)
        )

    return y_true, y_pred


model = MultiTCN()
model.to(DEVICE)

train_model(model, train_dl, epochs=10,device = DEVICE)


  0%|          | 0/10 [00:00<?, ?it/s]

Train loss: 1.145


 10%|█         | 1/10 [01:08<10:14, 68.23s/it]

Generalization Error: 1.465, Accuracy 0.317


 20%|██        | 2/10 [02:09<08:32, 64.04s/it]

...

Train loss: 1.066
Train loss: 1.049


 70%|███████   | 7/10 [07:32<03:13, 64.64s/it]

Generalization Error: 1.584, Accuracy 0.363


 80%|████████  | 8/10 [08:33<02:07, 63.51s/it]

Train loss: 1.048


In [ ]:
import gc

def free_cuda():
    gc.collect()
    torch.cuda.empty_cache()


free_cuda()
del model 

In [ ]:
y_true, y_pred = test_model(model, test_dl, DEVICE)

from sklearn.metrics import f1_score, classification_report

print(f1_score(y_true, y_pred, average="macro"))

print(classification_report(y_true, y_pred, target_names=["N1","N2","N3","R","W"]))

In [ ]:
np.argwhere(y_test == "N2").shape